# Binance daily price and USDT-volume descriptive analysis

This notebook validates the immutable canonical daily CSV files, prepares one Excel workbook for each of the 40 approved USDT-quoted pairs, and generates two cross-pair small-multiple figures. The analysis starts on `2019-07-01 00:00 UTC` or the pair's later first observed candle and ends on `2026-09-01 00:00 UTC`. No observations are interpolated, forward-filled, backfilled, or synthesized.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tempfile

import pandas as pd
from IPython.display import Image as NotebookImage, display


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / 'src'
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

from tcc_crypto.daily_price_volume_descriptive_analysis import (
    create_daily_small_multiple_figures,
    load_analysis_config,
    prepare_analysis_packets,
)

CONFIG_PATH = REPOSITORY_ROOT / 'configs/binance_daily_price_volume_descriptive_analysis.json'
PACKET_DIRECTORY = REPOSITORY_ROOT / 'tmp/daily_price_volume_workbook_inputs'
config, collection_config = load_analysis_config(CONFIG_PATH, repository_root=REPOSITORY_ROOT)
display(pd.DataFrame([
    ('Analysis ID', config.analysis_id),
    ('Pairs', len(collection_config.symbols)),
    ('Start (UTC)', config.analysis_start_utc.isoformat()),
    ('End (UTC)', config.analysis_end_utc.isoformat()),
    ('Output directory', str(config.output_directory)),
] , columns=['Field', 'Value']))


## Validate sources and prepare workbook inputs

The next cell validates timestamps, OHLC consistency, positivity, volume, coverage, and the approved time window before creating one JSON workbook input per pair.


In [ ]:
preparation_manifest = prepare_analysis_packets(
    CONFIG_PATH,
    repository_root=REPOSITORY_ROOT,
    packet_directory=PACKET_DIRECTORY,
)
display(preparation_manifest)


## Generate the cross-pair figures

The figure settings are stored in `configs/binance_daily_price_volume_descriptive_analysis.json`. The configured layout is an 8-by-5 grid in approved-symbol order, producing one panel for each of the 40 pairs. All panels share the configured UTC horizontal interval, while each pair uses an independent vertical scale so lower-priced and lower-volume assets remain visible.

For each daily candle, the price figure applies the natural logarithm pointwise:

$$y_{open,t} = \ln(open_t), \qquad y_{close,t} = \ln(close_t).$$

This is a transformation of positive price levels, not a return calculation. A negative plotted value means the original price was between zero and one USDT; it does not mean the asset had a negative price. Equal proportional price changes have equal vertical distances after the transformation.

The volume figure uses Binance `quote_asset_volume` without a logarithmic transformation. Values are divided by `1,000,000,000` only to display the linear scale in billions of USDT:

$$volume_{billions,t} = \frac{quote\_asset\_volume_t}{1{,}000{,}000{,}000}.$$

Before plotting, each observed series is reindexed to its daily UTC calendar. Missing dates become `NaN`, which breaks the plotted line. No price or volume is estimated for those dates, and dates before a pair's first observed candle remain blank.


In [ ]:
figure_artifacts = create_daily_small_multiple_figures(
    CONFIG_PATH,
    repository_root=REPOSITORY_ROOT,
)
figure_paths = pd.DataFrame([
    ('Natural-log open/close figure', figure_artifacts.natural_log_open_close_figure),
    ('USDT quote-volume figure', figure_artifacts.quote_volume_figure),
] , columns=['Figure', 'Path'])
display(figure_paths)
for figure_path in figure_paths['Path']:
    display(NotebookImage(filename=str(figure_path), width=1200))


### Reading the figures

The blue and orange lines in the price figure represent transformed opening and closing prices, respectively. Their frequent overlap is expected because daily opening and closing prices are often close relative to the full historical range. The volume figure shows the exchange-reported daily trading turnover in USDT, not holdings, market capitalization, net inflows, or transaction-level records.

Independent vertical scales make each pair's time pattern visible but prevent direct comparisons based only on the apparent height of lines in different panels. Use the labeled vertical ticks when comparing magnitudes. Re-running the cell regenerates the two PNG files under the configured output directory's `figures` folder; it does not modify the immutable raw CSV files.


## Build the Excel workbooks

Workbook generation uses the bundled `@oai/artifact-tool`. Set `TCC_CRYPTO_NODE` and `TCC_CRYPTO_NODE_MODULES` to the Node executable and bundled module directory before running this cell. The builder is copied to a temporary directory so the repository does not receive a generated `node_modules` link.


In [ ]:
node_executable = os.environ.get('TCC_CRYPTO_NODE')
node_modules = os.environ.get('TCC_CRYPTO_NODE_MODULES')
if not node_executable or not node_modules:
    raise RuntimeError('Set TCC_CRYPTO_NODE and TCC_CRYPTO_NODE_MODULES before workbook generation.')

builder_source = REPOSITORY_ROOT / 'scripts/build_daily_price_volume_workbooks.mjs'
qa_directory = REPOSITORY_ROOT / 'tmp/daily_price_volume_workbook_qa'
with tempfile.TemporaryDirectory(prefix='tcc_crypto_artifact_tool_') as temporary_directory:
    temporary_path = Path(temporary_directory)
    builder_path = temporary_path / builder_source.name
    shutil.copy2(builder_source, builder_path)
    (temporary_path / 'node_modules').symlink_to(Path(node_modules), target_is_directory=True)
    subprocess.run(
        [node_executable, str(builder_path), str(PACKET_DIRECTORY), str(config.output_directory), str(qa_directory)],
        cwd=REPOSITORY_ROOT,
        check=True,
    )

final_manifest = pd.read_csv(config.output_directory / 'analysis_manifest.csv')
display(final_manifest)


## Interpretation

The workbooks describe non-stationary price levels and exchange-reported activity aggregated inside daily klines. They do not estimate returns, risk, or transaction-level behavior. Comparisons require caution because listing dates, observed spans, missing daily openings, and partial semesters differ.
